# Trend Regression Models

Time-series regression can model the response as a function of time. The lecture begins with trend-only models:

No trend:

$$y_t = \beta_0 + \epsilon_t$$

Linear trend:

$$y_t = \beta_0 + \beta_1 t + \epsilon_t$$

Quadratic trend:

$$y_t = \beta_0 + \beta_1 t + \beta_2 t^2 + \epsilon_t$$

These are ordinary least squares models with time as a predictor. The usual regression assumptions still matter, especially independence of errors.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from checks import check_columns, check_no_missing

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True
cod = pd.read_csv(Path('data/cod_catch.csv'))
calc = pd.read_csv(Path('data/calculator_sales.csv'))
loan = pd.read_csv(Path('data/loan_requests.csv'))

## No-Trend Baseline

The no-trend model estimates one constant mean for all periods. For the cod catch example, this gives the lecture's baseline forecast: the sample average.

In [ ]:
cod_mean = smf.ols('Catch ~ 1', data=cod).fit()
print(cod_mean.summary().tables[1])
print(f'Mean forecast for any future month: {cod_mean.params["Intercept"]:.3f}')

## Linear Trend

A linear trend adds one period-to-period slope. It is useful when the plot shows a roughly steady increase or decrease.

In [ ]:
calc_linear = smf.ols('Sales ~ Time', data=calc).fit()
print(calc_linear.summary().tables[1])

future_calc = pd.DataFrame({'Time': [25, 26, 27]})
calc_pred = calc_linear.get_prediction(future_calc).summary_frame(alpha=0.05)
future_calc.join(calc_pred[['mean', 'mean_ci_lower', 'mean_ci_upper', 'obs_ci_lower', 'obs_ci_upper']]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(calc['Time'], calc['Sales'], marker='o', label='observed')
ax.plot(calc['Time'], calc_linear.fittedvalues, color='black', label='linear trend fit')
ax.set_xlabel('Time')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()

## Quadratic Trend

A quadratic trend can represent one smooth bend in the series. The lecture's loan request example has a strong upward pattern with curvature, so compare linear and quadratic fits.

In [ ]:
loan_linear = smf.ols('LoanRequests ~ Time', data=loan).fit()
loan_quad = smf.ols('LoanRequests ~ Time + I(Time**2)', data=loan).fit()
compare = pd.DataFrame({
    'model': ['linear', 'quadratic'],
    'R2_adj': [loan_linear.rsquared_adj, loan_quad.rsquared_adj],
    'AIC': [loan_linear.aic, loan_quad.aic],
    'residual_std_error': [np.sqrt(loan_linear.mse_resid), np.sqrt(loan_quad.mse_resid)],
})
compare.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loan['Time'], loan['LoanRequests'], marker='o', label='observed')
ax.plot(loan['Time'], loan_linear.fittedvalues, label='linear')
ax.plot(loan['Time'], loan_quad.fittedvalues, label='quadratic')
ax.set_xlabel('Time')
ax.set_ylabel('LoanRequests')
ax.legend()
plt.tight_layout()

Model choice should combine plots, fit statistics, residual diagnostics, and the forecasting context. A higher-order polynomial can fit historical data closely but may extrapolate poorly. Use the simplest model that captures the structure needed for the forecast horizon.